<a href="https://colab.research.google.com/github/Kavishka2401/GPTea/blob/C_1_Yield_Prediction/Machine_Learning_Pipeline/Final_Model_Preprocessing_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import Google Drive to get the final dataset
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Import Libraries
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import joblib

In [4]:
final_data_path = '/content/drive/MyDrive/DSGP_DOCS/Datasets/Final_Data_Merged.csv'

# Load final data
final_data = pd.read_csv(final_data_path)
df = final_data.copy()

## **Preprocessing**

### Remove missing values

In [6]:
df = df[df['YPH'].notna()].copy()

### Month mapping

In [7]:
month_map = {
    'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
    'July':7,'August':8,'September':9,'October':10,'November':11,'December':12
}
df['Month'] = df['Month'].map(month_map)

df = df.drop(columns=['Month'])

### Remove highly correlated and noisy features

In [8]:
df = df.drop(columns=[
    'temperature_2m_min',
    'temperature_2m_max',
    'relative_humidity_2m_min',
    'relative_humidity_2m_max',
    'wind_speed_10m_max',
    'precipitation_sum',
    'UT925',
    'BioShakthi',
    'Zinc',
    'sunshine_duration',
    'relative_humidity_2m_mean'
])

### Feature engineering

In [9]:
df['Dolomite_moisture'] = (
    df['Dolomite'] * (df['rain_sum'] - df['et0_fao_evapotranspiration'])
)

### Lag Features for Yield Per Hectare

In [10]:
for lag in [1, 12]:
    df[f'YPH_lag{lag}'] = df['YPH'].shift(lag).bfill()

### Lag Feature for Fertilizer

In [11]:
for fert in ['Compost']:
    df[f'{fert}_lag1'] = df[fert].shift(1).fillna(0)

### Define target & features

In [12]:
target = 'YPH'
features = [col for col in df.columns if col not in ['YPH', 'Year', 'Compost']]

### Define Training Window (3 Years)

In [13]:
WINDOW_YEARS = [2023, 2024, 2025]

df_final = df[df['Year'].isin(WINDOW_YEARS)].copy()

print("Training Years Used:", sorted(df_final['Year'].unique()))
print("Training Samples:", df_final.shape)

Training Years Used: [np.int64(2023), np.int64(2024), np.int64(2025)]
Training Samples: (36, 17)


## **Model Training**

In [14]:
# Alpha from validation
ALPHA = 5.0

X_final = df_final[features]
y_final = df_final[target]

# Model pipeline
final_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=ALPHA, random_state=42))
])

# Train on FULL 3-year window
final_pipeline.fit(X_final, y_final)

print("Final Ridge Model Trained Successfully on 2023-2025")

Final Ridge Model Trained Successfully on 2023-2025


## **Save Final Model**

In [15]:
model_path = "/content/drive/MyDrive/DSGP_DOCS/Final_Models/Final_Ridge_Model_23_24_25.pkl"
joblib.dump(final_pipeline, model_path)

print("Model saved successfully to Google Drive")

Model saved successfully to Google Drive
